#**Customer_Table**

#Reading_from_bronze_table

In [0]:
%python
df = spark.table('workspace.bronze.customers')
display(df)

customer_id,customer_zip_code_prefix,customer_city,customer_state
I74lXDOfoqsp,6020,goiania,GO
47TuLHF2s7X5,23020,viamao,RS
dQ0dqI8Qwlj8,75094,campinas,SP
iQCmWhNkIczb,89284,santana de parnaiba,SP
Dp2g6JH8tO5Z,39810,aripuana,MT
s18Fn5Tz7Jml,65790,sao jose do rio preto,SP
9rXAwNe0G0m5,25245,sao bento do sul,SC
o3yCAPMIPtLj,97560,sao paulo,SP
PHGrixjXAt4t,32430,barueri,SP
K6lNXbhXbvwg,8571,divinopolis,MG


#Data_Transformations

In [0]:
%sql
DESCRIBE workspace.bronze.customers;

col_name,data_type,comment
customer_id,string,null
customer_zip_code_prefix,int,null
customer_city,string,null
customer_state,string,null


#Data_quality_check

In [0]:
%sql
select count(*)as total_rows from workspace.bronze.customers;

total_rows
38279


In [0]:
%sql
select 
    count(*)-count(customer_id)as id_total_nulls,
    count(*)-count(customer_zip_code_prefix)as zip_total_nulls,
    count(*)-count(customer_city)as city_total_nulls,
    count(*)-count(customer_state) as state_total_nulls
from workspace.bronze.customers

id_total_nulls,zip_total_nulls,city_total_nulls,state_total_nulls
0,0,0,0


In [0]:
select customer_id,
count(*)as count
from workspace.bronze.customers
group by customer_id
having count(*)>1

customer_id,count


In [0]:
SELECT trim(customer_city),
    trim(customer_state)
FROM workspace.bronze.customers

trim(customer_city),trim(customer_state)
goiania,GO
viamao,RS
campinas,SP
santana de parnaiba,SP
aripuana,MT
sao jose do rio preto,SP
sao bento do sul,SC
sao paulo,SP
barueri,SP
divinopolis,MG


In [0]:
SELECT
    COUNT(CASE WHEN TRIM(customer_id) = '' THEN 1 END) AS empty_customer_id,
    COUNT(CASE WHEN TRIM(customer_city) = '' THEN 1 END) AS empty_city,
    COUNT(CASE WHEN TRIM(customer_state) = '' THEN 1 END) AS empty_state
FROM bronze.customers;

empty_customer_id,empty_city,empty_state
0,0,0


#Write_into_silver_table

In [0]:
%python
df = spark.table('workspace.bronze.customers')
df.write.mode("overwrite").saveAsTable("silver.customers")

#**Order_Items**

In [0]:
%python
df = spark.table('workspace.bronze.order_items')
display(df)

order_id,product_id,seller_id,price,shipping_charges
u6rPMRAYIGig,1slxdgbgWFax,3jwvL6ihC45G,24.1,20.9
ohY8f4FEbX19,77PgsiElQLeB,GlLj704QXlDB,42.89,12.28
I28liQek73i2,QVlD26X1y7NI,V3iKL8r9W9NR,50.21,67.11
bBG1T89mlY8W,yWlFGkKYfrpa,RNBdBKsXebna,89.1,62.05
CYxJJSQS8Lbo,h6MCbrwh5kiC,5Ja2lH0N2OZt,2139.99,9.41
kUkQCFPtDvrC,CApN1zdCu8Ad,smK689qrlIx3,84.55,20.65
eV98svHRmPNG,HOJj2KbHY8er,9hrosJpQLxRs,23.95,23.87
b2tsoISX5lnP,bKrL5OryV9HU,SSiPMxP6t9t9,692.0,44.59
O0D3th8M88nF,7ba9znpcLheP,WTbhLip6Y9IF,649.75,39.88
yBTGlSf8GGMV,TLhoqYaQLJ5g,ynoKalx03eHF,518.18,2.46


In [0]:
DESCRIBE workspace.bronze.order_items;

col_name,data_type,comment
order_id,string,null
product_id,string,null
seller_id,string,null
price,double,null
shipping_charges,double,null


In [0]:
select count(*)as total_rows from workspace.bronze.order_items;

total_rows
38279


In [0]:
select 
    count(*)-count(order_id)as order_id_total_nulls,
    count(*)-count(product_id)as pro_id_total_nulls,
    count(*)-count(seller_id)as seller_total_nulls,
    count(*)-count(price) as price_total_nulls,
    count(*)-count(shipping_charges) as shipping_total_nulls
from workspace.bronze.order_items

order_id_total_nulls,pro_id_total_nulls,seller_total_nulls,price_total_nulls,shipping_total_nulls
0,0,0,0,0


In [0]:
select order_id,
any_value(product_id) as product_id,
any_value(seller_id) as seller_id,
count(*)as count
from workspace.bronze.order_items
group by order_id
having count(*)>1

order_id,product_id,seller_id,count


In [0]:
SELECT *
FROM workspace.bronze.order_items
WHERE shipping_charges < 0
   OR price <= 0;

order_id,product_id,seller_id,price,shipping_charges


In [0]:
%python
df = spark.table('workspace.bronze.order_items')
df.write.mode("overwrite").saveAsTable("silver.order_items")

#**Orders_table**

In [0]:
DESCRIBE workspace.bronze.orders

col_name,data_type,comment
order_id,string,null
customer_id,string,null
order_purchase_timestamp,string,null
order_approved_at,string,null


In [0]:
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN order_purchase_timestamp IS NULL THEN 1 ELSE 0 END) AS null_order_purchase_timestamp,
    SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS null_order_approved_at

FROM workspace.bronze.orders;

total_rows,null_order_id,null_customer_id,null_order_purchase_timestamp,null_order_approved_at
38279,0,0,0,7


In [0]:
SELECT
    order_id,
    COUNT(*) AS cnt
FROM workspace.bronze.orders
GROUP BY order_id
HAVING COUNT(*) > 1;

order_id,cnt


In [0]:
CREATE OR REPLACE TABLE workspace.silver.orders AS

SELECT
    *,
    
    to_timestamp(
        order_purchase_timestamp,
        'M/d/yyyy H:mm'
    ) AS order_purchase_timestamp_new,

    to_timestamp(
        order_approved_at,
        'M/d/yyyy H:mm'
    ) AS order_approved_at_new

FROM workspace.bronze.orders;

num_affected_rows,num_inserted_rows


In [0]:
DESCRIBE workspace.silver.orders;

col_name,data_type,comment
order_id,string,null
customer_id,string,null
order_purchase_timestamp,string,null
order_approved_at,string,null
order_purchase_timestamp_new,timestamp,null
order_approved_at_new,timestamp,null


هل فيه Order تابع لـCustomer مش موجود في Customers؟

In [0]:
SELECT o.*
FROM workspace.bronze.orders o
LEFT JOIN workspace.bronze.customers c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

order_id,customer_id,order_purchase_timestamp,order_approved_at


In [0]:
%python
df = spark.table('workspace.bronze.orders')
df.write.mode("overwrite").saveAsTable("silver.orders")

#Payments_table

In [0]:
DESCRIBE workspace.bronze.payments

col_name,data_type,comment
order_id,string,null
payment_sequential,int,null
payment_type,string,null
payment_installments,int,null
payment_value,double,null


In [0]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN payment_sequential IS NULL THEN 1 ELSE 0 END) AS null_payment_sequential,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS null_payment_type,
    SUM(CASE WHEN payment_installments IS NULL THEN 1 ELSE 0 END) AS null_payment_installments,
    SUM(CASE WHEN payment_value IS NULL THEN 1 ELSE 0 END) AS null_payment_value
FROM workspace.bronze.payments;

total_rows,null_order_id,null_payment_sequential,null_payment_type,null_payment_installments,null_payment_value
38279,0,0,0,0,0


In [0]:
SELECT
    payment_type,
    COUNT(*) AS count
FROM workspace.bronze.payments
GROUP BY payment_type
ORDER BY count DESC;

payment_type,count
credit_card,28265
wallet,7432
voucher,2090
debit_card,492


In [0]:
SELECT
    order_id,
    payment_sequential,
    LOWER(TRIM(payment_type)) AS payment_type,
    payment_installments,
    payment_value
FROM workspace.bronze.payments;

order_id,payment_sequential,payment_type,payment_installments,payment_value
u6rPMRAYIGig,1,credit_card,2,155.77
ohY8f4FEbX19,1,credit_card,1,4.07
I28liQek73i2,1,wallet,1,381.59
bBG1T89mlY8W,1,credit_card,3,14.76
CYxJJSQS8Lbo,1,wallet,1,284.09
kUkQCFPtDvrC,1,credit_card,2,342.02
eV98svHRmPNG,1,credit_card,5,48.71
b2tsoISX5lnP,1,credit_card,1,204.4
O0D3th8M88nF,1,credit_card,3,997.37
yBTGlSf8GGMV,1,credit_card,3,204.62


In [0]:
SELECT *
FROM workspace.bronze.payments
WHERE payment_installments <= 0;

order_id,payment_sequential,payment_type,payment_installments,payment_value


In [0]:
SELECT *
FROM workspace.bronze.payments
WHERE payment_value < 0;

order_id,payment_sequential,payment_type,payment_installments,payment_value


In [0]:
SELECT p.*
FROM workspace.bronze.payments p
LEFT JOIN workspace.bronze.orders o
    ON p.order_id = o.order_id
WHERE o.order_id IS NULL;

order_id,payment_sequential,payment_type,payment_installments,payment_value


In [0]:
%python
df = spark.table('workspace.bronze.payments')
df.write.mode("overwrite").saveAsTable("silver.payments")

#Products_Table

In [0]:
DESCRIBE workspace.bronze.products

col_name,data_type,comment
product_id,string,null
product_category_name,string,null
product_weight_g,int,null
product_length_cm,int,null
product_height_cm,int,null
product_width_cm,int,null


In [0]:
SELECT COUNT(*) AS null_product_id
FROM workspace.bronze.products
WHERE product_id IS NULL;

null_product_id
0


In [0]:
SELECT
    product_id,
    COUNT(*) AS count
FROM workspace.bronze.products
GROUP BY product_id
HAVING COUNT(*) > 1;

product_id,count
1slxdgbgWFax,3
77PgsiElQLeB,2
QVlD26X1y7NI,3
yWlFGkKYfrpa,6
h6MCbrwh5kiC,4
CApN1zdCu8Ad,2
HOJj2KbHY8er,4
7ba9znpcLheP,2
TLhoqYaQLJ5g,4
EG4wDSpFyTth,93


In [0]:
SELECT DISTINCT product_category_name
FROM workspace.bronze.products
ORDER BY product_category_name;

product_category_name
null
agro_industry_and_commerce
air_conditioning
art
arts_and_craftmanship
audio
auto
baby
bed_bath_table
books_general_interest


In [0]:
SELECT *
FROM workspace.bronze.products
WHERE product_weight_g <= 0;

product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
aMbBkxpim3xt,toys,0,30,25,30
aMbBkxpim3xt,toys,0,30,25,30


In [0]:
SELECT *
FROM workspace.bronze.products
WHERE product_length_cm <= 0
   OR product_height_cm <= 0
   OR product_width_cm <= 0;

product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In [0]:
SELECT
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
    SUM(CASE WHEN product_category_name IS NULL THEN 1 ELSE 0 END) AS null_category,
    SUM(CASE WHEN product_weight_g IS NULL THEN 1 ELSE 0 END) AS null_weight,
    SUM(CASE WHEN product_length_cm IS NULL THEN 1 ELSE 0 END) AS null_length,
    SUM(CASE WHEN product_height_cm IS NULL THEN 1 ELSE 0 END) AS null_height,
    SUM(CASE WHEN product_width_cm IS NULL THEN 1 ELSE 0 END) AS null_width
FROM workspace.bronze.products;

null_product_id,null_category,null_weight,null_length,null_height,null_width
0,168,10,10,10,10


In [0]:
CREATE OR REPLACE TABLE silver.products AS
SELECT
    product_id,
    product_category_name,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY product_id
               ORDER BY product_id
           ) AS rn
    FROM bronze.products
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%python
df = spark.table('workspace.bronze.products')
df.write.mode("overwrite").saveAsTable("silver.products")